# Day 6 | ILT 1: Business Process Mapping & Star Schema (Kimball)
### GlobalMart Data Engineering Bootcamp
---
**Duration:** 60 min &nbsp;|&nbsp; **Level:** Intermediate &nbsp;|&nbsp; **Tags:** dimensional-modeling, kimball, star-schema, fact-vs-dimension

---
**Goal:** Understand why data warehouses are modeled around business processes, and use GlobalMart's sale process to introduce the star schema shape that `fact_sales` will follow.

---
**INSTRUCTOR NOTE:**
Students just finished building the Silver layer (Day 5) for all 4 GlobalMart sources. Silver is clean, row-level data — but it is still shaped like the source systems, not like something a business user could query. Today we introduce the shape it needs to become: a **star schema**. This session is conceptual — no code. The hands-on later this week (Day 6 HOL 1, then Day 7's Gold build) is where students actually build it.

## Learning Objectives

By the end of this session, students will be able to:

1. Explain what a **business process** is, in the data-warehousing sense
2. Explain why Kimball's methodology models a warehouse **one business process at a time**, not one report at a time
3. Distinguish a **fact** from a **dimension** using plain-language definitions
4. Identify GlobalMart's core business process and describe the star schema it produces
5. Explain why `fact_sales` sits at the center of that star, surrounded by its 6 dimensions

---
## Section 1: What Is a Business Process?

**INSTRUCTOR NOTE:**
Open with this question: *"If I asked you to build a report for GlobalMart's finance team, and a separate report for the marketing team, would you build two separate databases?"* Let students answer. The point: you would NOT — you would build the warehouse around the underlying **business process** (a sale happening), and let both teams query the same fact table differently.

---

A **business process** is a repeatable, measurable activity that happens in the business — something that generates events you can count, sum, or track over time.

| Not a business process | A business process |
|---|---|
| "Sales performance" (too vague — performance according to whom?) | **A sale happening** (customer buys N units of a product, at a price, on a date) |
| "Customer satisfaction" (an opinion, not an event) | **A return being filed** (a measurable event with a date, product, and reason) |
| "Marketing effectiveness" (a judgment) | **A marketing campaign being sent and opened** (a measurable event) |

### Why model around the *process*, not the *report*

Kimball's core rule: **build the warehouse one business process at a time**, not one report at a time. If you instead built a separate table for every report someone asked for ("revenue by region" table, "revenue by category" table, "top customers" table...), you would end up with dozens of overlapping, inconsistent tables — exactly the **Data Silos** and **Unclear Origin** problems from GlobalMart's original 5 problems (Day 1).

Instead: model the **sale** process once, correctly, as `fact_sales`. Every report — revenue by region, top customers, category performance — is just a different `GROUP BY` on the *same* trusted table.

> This is exactly why `fact_sales` was introduced on Day 1 as **the** deliverable of this entire course, not **a** deliverable among many.

---
## Section 2: Facts vs. Dimensions

**INSTRUCTOR NOTE:**
Use the classic Kimball one-liner: *"Facts are verbs, dimensions are nouns."* A sale **happens** (verb, fact) *to* a customer, *of* a product, *on* a date, *shipped to* an address, *paid by* a method, *as part of* an order (all nouns, dimensions).

---

| | Fact | Dimension |
|---|---|---|
| **What it represents** | A measurable business event | The context describing that event |
| **Contains** | Numbers you SUM / AVG / COUNT ("measures") + foreign keys to dimensions | Descriptive attributes you filter/group by |
| **Grows** | Fast — one new row per event (every order line item) | Slowly — a customer or product doesn't change every day |
| **GlobalMart example** | `fact_sales` — one row per order line item | `dim_customer`, `dim_product`, `dim_date`, `dim_address`, `dim_payment_method`, `dim_orders` |
| **Typical question it answers alone** | "How much did we sell?" (needs a dimension to mean anything specific) | "Who is this customer?" (needs a fact to be interesting to the business) |

### Neither is useful alone

`fact_sales` with no dimensions is just a pile of numbers — "&#8377;14,392,000 in `Sales_amount`" means nothing without knowing *whose* sales, *which* products, *when*. `dim_customer` with no fact table is just a phone book — accurate, but it doesn't tell the business anything about revenue.

**The value is in the join.** `fact_sales JOIN dim_product` lets you ask "which category sells the most?" — a question neither table can answer by itself.

---
## Section 3: GlobalMart's Business Process

**INSTRUCTOR NOTE:**
This is the payoff moment for the whole course narrative. Point back to the architecture diagram from Day 1 — every ingestion pipeline, every Bronze/Silver table, every CDC and Autoloader job built so far exists **only** to make this one process measurable correctly.

---

GlobalMart's core business process is: **a customer buys product(s) in an order.**

Every time that happens, GlobalMart's systems record it across multiple source tables — `orders`, `order_items`, `products`, `customers`, `addresses`, `payments`, `payment_methods`. Right now (after Day 5), that data is clean and trustworthy in Silver — but it is still scattered across 7 separate tables, shaped like the source system, not like a business question.

**The star schema's job:** collapse that scattered, source-shaped data into one fact table (`fact_sales`) surrounded by the dimensions that give it meaning.

> `fact_sales`'s grain, measures (`Quantity_purchased`, `Actual_price`, `Discounted_price`, `Sales_amount`), and 6 dimensions were already named back on Day 1 as this course's central deliverable. Today's job is to understand *why* that shape is the right one — grain and dimension design get their full rigorous treatment in ILT 2, right after this session.

---
## Section 4: The Star Schema Shape

**INSTRUCTOR NOTE:**
Draw this on the board. The visual of a literal "star" — one table in the middle, dimensions radiating out — is why it's called a star schema. Contrast briefly with a "snowflake schema" (dimensions further normalized into sub-dimensions) — GlobalMart intentionally does NOT snowflake; every dimension here is a single flat table, which keeps joins simple and fast.

---

```
                dim_date              dim_orders
                     \                    /
   dim_customer ---- fact_sales ---- dim_product
                     /          \
             dim_address    dim_payment_method
```

| Table | Role | Grain / Contents |
|---|---|---|
| **fact_sales** | Center of the star | One row per order line item |
| `dim_customer` | Who bought it | One row per customer version (SCD2) |
| `dim_product` | What was sold | One row per product version (SCD2) |
| `dim_date` | When | One row per calendar day — generated, not sourced from Silver |
| `dim_address` | Where it shipped | One row per address |
| `dim_payment_method` | How it was paid | One row per payment method (~5 rows total) |
| `dim_orders` | Order header context | One row per order (channel, shipping tier, supplier) |

> Six dimensions — matching the Day 1 spec's spirit, with one addition: `dim_orders` for order-header attributes. ILT 2 (next) designs each one in depth: natural key, surrogate key, and which ones are realistic candidates for SCD Type 2 (history tracking). Two of them — `dim_customer` and `dim_product` — actually already carry that history by the time you get to them, because Silver (Day 5) built it in; today's hands-on republishes it in Gold, it doesn't build it from scratch.

---
## Section 5: What's Next

| Session | What Happens |
|---|---|
| **Day 6 ILT 2 (next, today)** | Grain definition — the single most important decision before building anything — then a deep dive into designing each of the 6 dimensions |
| **Day 6 ILT 3 (today)** | Fact table types, additive vs. non-additive measures, degenerate dimensions, and why `fact_sales` references some dimensions by natural key instead of surrogate key |
| **Day 6 HOL 1 (today)** | Hands-on: design the schema on paper, then build all 6 dimension tables for real in `<your-catalog>.gold` |
| **Day 7** | Build `fact_sales` itself, joining the grain (`order_items`) against the dimensions you build today |

**INSTRUCTOR NOTE:**
Close with: *"Everything from Day 1 through Day 5 — CDC, Autoloader, Bronze, Silver — was infrastructure. Today is where the business value actually gets shaped. This is the payoff."*